In [31]:
import pandas as pd
import requests
import os
from dotenv import load_dotenv



load_dotenv(override=True)
api_key = os.getenv("api_key")

def get_release_year(imdbID):
    """Obtem a data de lançamento do filme por meio da consulta à API do IMDB."""
    try:
        url = f"http://www.omdbapi.com/?i={imdbID}&apikey={api_key}"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        return data.get("Released")
    except Exception as e:
        print(f"Erro ao buscar dados da API:{e}")
        raise


def get_data():
    """Faz a extração dos dados do arquivo .csv e transforma em um dataframe do Pandas."""
    path = r"C:\Projetos python\Limpeza_dados-Pandas\Messy_data\messy_IMDB_dataset.csv"
    df = pd.read_csv(path,sep=';',encoding='latin-1')
    return df


In [32]:
# Exploração dos dados:

df = get_data()

df.info()

df.describe()

print(df.nunique())

df.dropna(how='all')

df = df.drop('Unnamed: 8',axis = 1)

df.drop_duplicates()

display(df)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   IMBD title ID   100 non-null    object 
 1   Original titlÊ  100 non-null    object 
 2   Release year    100 non-null    object 
 3   Genrë¨          100 non-null    object 
 4   Duration        99 non-null     object 
 5   Country         100 non-null    object 
 6   Content Rating  77 non-null     object 
 7   Director        100 non-null    object 
 8   Unnamed: 8      0 non-null      float64
 9   Income          100 non-null    object 
 10   Votes          100 non-null    object 
 11  Score           100 non-null    object 
dtypes: float64(1), object(11)
memory usage: 9.6+ KB
IMBD title ID     100
Original titlÊ    100
Release year       99
Genrë¨             59
Duration           71
Country            18
Content Rating      7
Director           64
Unnamed: 8          0
Income        

,IMBD title ID,Original titlÊ,Release year,Genrë¨,Duration,Country,Content Rating,Director,Income,Votes,Score
0,tt0111161,The Shawshank Redemption,1995-02-10,Drama,142,USA,R,Frank Darabont,$ 28815245,2.278.845,9.3
1,tt0068646,The Godfather,09 21 1972,"Crime, Drama",175,USA,R,Francis Ford Coppola,$ 246120974,1.572.674,9.2
2,tt0468569,The Dark Knight,23 -07-2008,"Action, Crime, Drama",152,US,PG-13,Christopher Nolan,$ 1005455211,2.241.615,9.
3,tt0071562,The Godfather: Part II,1975-09-25,"Crime, Drama",220,USA,R,Francis Ford Coppola,"$ 4o8,035,783",1.098.714,"9,.0"
4,tt0110912,Pulp Fiction,1994-10-28,"Crime, Drama",,USA,R,Quentin Tarantino,$ 222831817,1.780.147,"8,9f"
...,...,...,...,...,...,...,...,...,...,...,...
96,tt0070735,The Sting,1974-03-21,"Comedy, Crime, Drama",129,USA,PG,George Roy Hill,$ 156000000,236.285,7.5
97,tt0082096,Das Boot,1982-03-18,"Adventure, Drama, Thriller",149,West Germany,R,Wolfgang Petersen,$ 11487676,226.427,7.5
98,tt0059578,Per qualche dollaro in piÃ¹,1965-12-20,Western,132,Italy,NaN,Sergio Leone,$ 15000000,226.039,7.4
99,tt1832382,Jodaeiye Nader az Simin,2011-10-21,Drama,123,Iran,PG-13,Asghar Farhadi,$ 22926076,214.165,7.4


In [33]:

# Tratamento de coluna do tempo de duração do filme:

df = df.copy()

df["Duration"]= df["Duration"].astype(str).str.strip()
df["Duration"] = df["Duration"].str.replace(r'[^0-9]','',regex=True)

df["Duration"].unique()

df[df["Duration"] == ""]
df["Duration"] = df["Duration"].replace('', pd.NA)
df["Duration"] = pd.to_numeric(df["Duration"], errors='coerce')



df["Duration"] = df["Duration"].fillna(
    df.groupby("Genrë¨")["Duration"].transform("median")
)



In [34]:

# Tratamento da coluna de paises do filme

paises = {'USA':'United States', 'US.':'United States', 'US': 'United States','Italy1':'Italy',
          'UK':'United Kingdom','New Zesland':'New Zeland','New Zealand':'New Zeland'}

for i,j in paises.items():
    df["Country"] = df["Country"].str.replace(i,j)

df.sample(20)



,IMBD title ID,Original titlÊ,Release year,Genrë¨,Duration,Country,Content Rating,Director,Income,Votes,Score
52,tt0209144,Memento,2001-01-19,"Mystery, Thriller",113.0,United States,R,Christopher Nolan,$ 39970386,1.098.879,8.2
9,tt0137523,Fight Club,10-29-99,Drama,122.0,United Kingdom,R,David Fincher,$ 101218804,1.807.440,8.8
67,tt0119698,Mononoke-hime,2000-05-19,"Animation, Adventure, Fantasy",134.0,Japan,PG-13,Hayao Miyazaki,$ 169785629,331.045,8.0
16,tt0080684,Star Wars: Episode V - The Empire Strikes Back,1980-09-19,"Action, Adventure, Fantasy",126.0,United States,PG,Irvin Kershner,$ 549265501,1.132.073,"8,7e-0"
83,tt0086250,Scarface,1984-02-34,"Crime, Drama",170.0,United States,R,Brian De Palma,$ 66023585,721.343,7.8
88,tt0062622,2001: A Space Odyssey,1968-12-12,"Adventure, Sci-Fi",149.0,United Kingdom,G,Stanley Kubrick,$ 68989547,587.866,7.6
0,tt0111161,The Shawshank Redemption,1995-02-10,Drama,142.0,United States,R,Frank Darabont,$ 28815245,2.278.845,9.3
28,tt6751668,Gisaengchung,2019-11-07,"Comedy, Drama, Thriller",132.0,South Korea,NaN,Bong Joon Ho,$ 257604912,470.931,8.6
2,tt0468569,The Dark Knight,23 -07-2008,"Action, Crime, Drama",152.0,United States,PG-13,Christopher Nolan,$ 1005455211,2.241.615,9.
24,tt0120689,The Green Mile,2000-10-03,"Crime, Drama, Fantasy",189.0,United States,R,Frank Darabont,$ 286801374,1.112.336,8.6


In [35]:

# Tratamento da coluna de data de lancamento do filme.

df["Release year"] = df["Release year"].astype(str).str.strip().str.lower()
df["Release year"] = df["Release year"].str.replace(r'\s+', ' ', regex=True)

df["Release year"] = df["Release year"].str.replace(r'(\d+)(st|nd|rd|th)', r'\1', regex=True)
df["Release year"] = df["Release year"].str.replace(r'\b(of|year|the)\b', '', regex=True)
df["Release year"] = df["Release year"].str.replace(r'\s+', ' ', regex=True).str.strip()

meses = {
    'january': '01', 'february': '02', 'march': '03', 'april': '04',
    'may': '05', 'june': '06', 'july': '07', 'august': '08',
    'september': '09', 'october': '10', 'november': '11', 'december': '12',
    'enero': '01', 'febrero': '02', 'marzo': '03', 'abril': '04',
    'mayo': '05', 'junio': '06', 'julio': '07', 'agosto': '08',
    'septiembre': '09', 'octubre': '10', 'noviembre': '11', 'diciembre': '12'
}

for nome, numero in meses.items():
    df["Release year"] = df["Release year"].str.replace(nome, numero, regex=False)

df["Release year"] = df["Release year"].str.replace(r'[/\s]', '-', regex=True)
df["Release year"] = df["Release year"].str.replace(r'-+', '-', regex=True).str.strip('-')

meses_abrev = {
    'jan': '01', 'feb': '02', 'mar': '03', 'apr': '04',
    'may': '05', 'jun': '06', 'jul': '07', 'aug': '08',
    'sep': '09', 'oct': '10', 'nov': '11', 'dec': '12'
}

for nome, numero in meses_abrev.items():
    df["Release year"] = df["Release year"].str.replace(nome, numero, regex=False)


df["Release year"] = df["Release year"].str.replace(',', '', regex=False)

teste = pd.to_datetime(df["Release year"], errors='coerce')


tentativa2 = pd.to_datetime(df["Release year"], errors='coerce', dayfirst=True)
df["Release year"] = teste.combine_first(tentativa2)









In [36]:

# Limpando coluna de receita

df["Income"] = df["Income"].str.replace(r'$|[^0-9]','',regex=True)

df.loc[df["Income"].isnull()]

df = df.dropna(how='all')

print(df["Income"].unique())

df["Income"] = pd.to_numeric(df["Income"],errors='coerce')
df.sample(10)

['28815245' '246120974' '1005455211' '48035783' '222831817' '1142271098'
 '322287794' '576' '869784991' '101218804' '678229452' '887934303'
 '25252481' '465718588' '951227416' '549265501' '46879633' '108997629'
 '696742056' '327333559' '272753884' '775768912' '482349603' '286801374'
 '30680793' '355467056' '230098753' '257604912' '6130720' '322773'
 '465361176' '291465034' '109676311' '388774684' '23875127' '19552639'
 '520884847' '23341568' '968511805' '1074251311' '426588510' '120072577'
 '48983260' '32008644' '4374761' '112911' '516962' '13826605' '457688'
 '1081133191' '425368238' '39970386' '521311860' '46520613' '390133212'
 '2048359754' '108110316' '2797800564' '91968688' '15002116' '9443876'
 '37032034' '807083670' '77356942' '375540831' '60262836' '169785629'
 '5472914' '969879' '299645' '321455689' '356296601' '213216216'
 '475347111' '2889963' '74036715' '404265438' '225933435' '83557872'
 '7390108' '26903440' '1066969703' '66023585' '28441292' '173924742'
 '46357676' '13138

,IMBD title ID,Original titlÊ,Release year,Genrë¨,Duration,Country,Content Rating,Director,Income,Votes,Score
24,tt0120689,The Green Mile,2000-10-03,"Crime, Drama, Fantasy",189.0,United States,R,Frank Darabont,286801374,1.112.336,8.6
98,tt0059578,Per qualche dollaro in piÃ¹,1965-12-20,Western,132.0,Italy,NaN,Sergio Leone,15000000,226.039,7.4
85,tt0211915,Le fabuleux destin d'AmÃ©lie Poulain,2002-01-25,"Comedy, Romance",122.0,France,R,Jean-Pierre Jeunet,173924742,690.480,7.7
7,tt0050083,12 Angry Men,1957-09-04,"Crime, Drama",96.0,United States,Not Rated,Sidney Lumet,576,668.473,8.9
15,tt0167261,The Lord of the Rings: The Two Towers,NaT,"Action, Adventure, Drama",179.0,New Zeland,PG-13,Peter Jackson,951227416,1.449.778,8.7.
68,tt0087843,Once Upon a Time in America,1984-09-28,"Crime, Drama",229.0,United States,R,Sergio Leone,5472914,302.317,8.0
79,tt0208092,Snatch,2001-03-16,"Comedy, Crime",104.0,United Kingdom,R,Guy Ritchie,83557872,766.589,7.8
3,tt0071562,The Godfather: Part II,1975-09-25,"Crime, Drama",220.0,United States,R,Francis Ford Coppola,48035783,1.098.714,"9,.0"
96,tt0070735,The Sting,1974-03-21,"Comedy, Crime, Drama",129.0,United States,PG,George Roy Hill,156000000,236.285,7.5
56,tt4154756,Avengers: Infinity War,2018-04-25,"Action, Adventure, Sci-Fi",149.0,United States,NaN,"Anthony Russo, Joe Russo",2048359754,796.486,8.2


In [37]:
df = df.drop(columns=["Content Rating"])


In [38]:
# # Tratamento da coluna de score e renomeando colunas.

df = df.rename(columns={"IMBD title ID":"ID_movie","Original titlÊ":"original_title","Release year":"release_year",
"Genrë¨":"genre"," Votes ":"votes","Duration":"duration","Income":"income","Score":"score","Country":"country","Director":"director"})

df["score"] = df["score"].str.extract(r"(\d+(?:\.\d+)?)")

df["score"] = pd.to_numeric(df["score"])

df["score"].value_counts()



score
8.0    11
8.6    11
8.3     8
8.2     8
8.4     7
8.1     7
7.5     6
8.5     6
7.8     6
7.9     5
7.6     4
7.7     4
8.7     4
7.4     3
8.8     3
8.9     3
9.0     2
9.2     1
9.3     1
Name: count, dtype: int64

In [39]:
# Tratando coluna de votos

df["votes"] = df["votes"].str.replace('.','').str.strip()
df["votes"].unique()
df["votes"] = pd.to_numeric(df["votes"])


In [40]:
# Substituindo os nulos com dados da API do IMDB.

mask_missing = df["release_year"].isna()

df["release_year_origem"] = "csv_original"

df.loc[mask_missing,"release_year_origem"] = "api_omdb"


df.loc[mask_missing, "release_year"] = pd.to_datetime(

    df.loc[mask_missing, "ID_movie"].apply(get_release_year)
    
    )

display(df)

# df.isnull().sum()


,ID_movie,original_title,release_year,genre,duration,country,director,income,votes,score,release_year_origem
0,tt0111161,The Shawshank Redemption,1995-02-10,Drama,142.0,United States,Frank Darabont,28815245,2278845,9.3,csv_original
1,tt0068646,The Godfather,1972-03-24,"Crime, Drama",175.0,United States,Francis Ford Coppola,246120974,1572674,9.2,api_omdb
2,tt0468569,The Dark Knight,2008-07-18,"Action, Crime, Drama",152.0,United States,Christopher Nolan,1005455211,2241615,9.0,api_omdb
3,tt0071562,The Godfather: Part II,1975-09-25,"Crime, Drama",220.0,United States,Francis Ford Coppola,48035783,1098714,9.0,csv_original
4,tt0110912,Pulp Fiction,1994-10-28,"Crime, Drama",170.0,United States,Quentin Tarantino,222831817,1780147,8.0,csv_original
...,...,...,...,...,...,...,...,...,...,...,...
96,tt0070735,The Sting,1974-03-21,"Comedy, Crime, Drama",129.0,United States,George Roy Hill,156000000,236285,7.5,csv_original
97,tt0082096,Das Boot,1982-03-18,"Adventure, Drama, Thriller",149.0,West Germany,Wolfgang Petersen,11487676,226427,7.5,csv_original
98,tt0059578,Per qualche dollaro in piÃ¹,1965-12-20,Western,132.0,Italy,Sergio Leone,15000000,226039,7.4,csv_original
99,tt1832382,Jodaeiye Nader az Simin,2011-10-21,Drama,123.0,Iran,Asghar Farhadi,22926076,214165,7.4,csv_original


In [41]:
#Carregando arquivo limpo e tratado.


df.to_csv('Cleaned_data/imbd_dataset_cleaned.csv',index=False)